In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report, f1_score

from lightgbm import LGBMClassifier
from sklearn.multioutput import MultiOutputClassifier, ClassifierChain

In [2]:
df = pd.read_csv("ClaimDenialInputMultiLabel.csv")

C:\Users\SreejaChiluveru\AppData\Local\Temp\ipykernel_3744\3972365444.py:1: DtypeWarning: Columns (18,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("ClaimDenialInputMultiLabel.csv")


In [4]:
df.shape
df.info()
df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 39 columns):
 #   Column                   Non-Null Count    Dtype  
---  ------                   --------------    -----  
 0   Clinic                   1000000 non-null  object 
 1   TPCLIID                  1000000 non-null  object 
 2   LIATPCLIid               926015 non-null   object 
 3   ServiceDt                1000000 non-null  object 
 4   Service                  1000000 non-null  object 
 5   ClaimID                  1000000 non-null  int64  
 6   AmountCharged            1000000 non-null  float64
 7   CPTCode                  1000000 non-null  object 
 8   ClientID                 1000000 non-null  object 
 9   ClaimBillDate            1000000 non-null  object 
 10  Payer                    1000000 non-null  object 
 11  Provider                 999988 non-null   object 
 12  BillingProviderNPI       1000000 non-null  object 
 13  ClaimFacilityNPI         1000000 non-null  

,Clinic,TPCLIID,LIATPCLIid,ServiceDt,Service,ClaimID,AmountCharged,CPTCode,ClientID,ClaimBillDate,...,CoIns,CltResp,Balance,MultiFlag,SameDayCli,DaysBetServiceToBilling,tpcliStrModifier,tpcliStrPOS,f21diag1,f11insdob
0,CLN_13838227,TP_41935074,LTP_1870199,2025-06-22,Methadone Maintenance Week,143008644,297.61,H0020,CLT_85822412,2025-04-06,...,0.0,0.0,0.00,N,0,1,NaN,11,F1120,NaN
1,CLN_13838227,TP_44604858,LTP_82010373,2025-06-29,Methadone Maintenance Week,143008720,208.74,H0020,CLT_85822412,2025-04-13,...,0.0,0.0,0.00,N,0,1,NaN,11,F1120,NaN
2,CLN_13838227,TP_26594858,NaN,2025-07-06,Methadone Maintenance Week,143009390,238.49,H0020,CLT_85822412,2025-06-03,...,0.0,0.0,224.01,NaN,0,45,NaN,11,F1120,NaN
3,CLN_13838227,TP_95683641,LTP_2144057,2025-07-13,Methadone Maintenance Week,143008875,245.26,H0020,CLT_85822412,2025-04-27,...,0.0,0.0,283.10,Z,0,1,NaN,11,F1120,NaN
4,CLN_13838227,TP_90042682,LTP_38875939,2025-07-20,Methadone Maintenance Week,143008961,228.32,H0020,CLT_85822412,2025-05-05,...,0.0,0.0,226.06,Z,0,2,NaN,11,F1120,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,CLN_20041488,TP_28231781,LTP_22240665,2025-10-19,Take Home Dose MMT,105186924,6.70,H0020,CLT_82150984,2025-04-27,...,0.0,0.0,0.00,N,0,2,NaN,11,F1120,NaN
999996,CLN_20041488,TP_14603567,LTP_86430893,2025-10-25,Take Home Dose MMT,105187299,8.68,H0020,CLT_82150984,2025-05-04,...,0.0,0.0,0.00,N,0,3,NaN,11,F1120,NaN
999997,CLN_20041488,TP_67501817,LTP_63741515,2025-10-26,Take Home Dose MMT,105187299,7.39,H0020,CLT_82150984,2025-05-04,...,0.0,0.0,0.00,N,0,2,NaN,11,F1120,NaN
999998,CLN_20041488,TP_83554380,LTP_50442122,2025-11-02,Take Home Dose MMT,105187791,8.79,H0020,CLT_82150984,2025-05-11,...,0.0,0.0,0.00,N,0,2,NaN,11,F1120,NaN


using sample data for eda  and creating delay

In [5]:
df_sample = df.sample(100000, random_state=42)

fixing dates

In [7]:
df_sample['ServiceDt'] = pd.to_datetime(df_sample['ServiceDt'])
df_sample['ClaimBillDate'] = pd.to_datetime(df_sample['ClaimBillDate'])

df_sample['delay'] = (df_sample['ClaimBillDate'] - df_sample['ServiceDt']).dt.days

# remove invalid
df_sample = df_sample[df_sample['delay'] >= 0]

target eda


In [8]:
targets = ['target1', 'target2', 'target3', 'target4']

df_sample['labels'] = df_sample[targets].values.tolist()
df_sample['labels'] = df_sample['labels'].apply(lambda x: [i for i in x if pd.notna(i)])

In [15]:
all_labels = pd.Series([l for sub in df_sample['labels'] for l in sub])
label_counts = all_labels.value_counts()

label_counts.head(20)

23.0    400
8.0     282
32.0    235
14.0    196
5.0     155
31.0    119
2.0      77
25.0     50
3.0      48
21.0     41
19.0     34
16.0     32
Name: count, dtype: int64

ACTION REQUIRED FOR TTHIS ONE

In [16]:
min_support = 30  # higher because 1M data

valid_labels = label_counts[label_counts >= min_support].index

df_sample['labels'] = df_sample['labels'].apply(lambda x: [i for i in x if i in valid_labels])

LABEL CARDINALITY

In [17]:
df_sample['label_count'] = df_sample['labels'].apply(len)

df_sample['label_count'].value_counts()

label_count
0    2969
1    1228
2     184
3      23
4       1
Name: count, dtype: int64

STEP 5: Co-occurrence (CRITICAL)

In [18]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
Y_temp = mlb.fit_transform(df_sample['labels'])

co_matrix = np.dot(Y_temp.T, Y_temp)

In [23]:
targets = ['target1', 'target2', 'target3', 'target4']

# create labels
df['labels'] = df[targets].values.tolist()
df['labels'] = df['labels'].apply(lambda x: [i for i in x if pd.notna(i)])

# NOW create label_count
df['label_count'] = df['labels'].apply(len)

In [24]:
df_sample['CPTCode'].value_counts().head()

CPTCode
H0020    2003
S0109     707
80305     429
G2067     191
80307     152
Name: count, dtype: int64

ACTION FOR ABOVE OEN

In [25]:
cpt_rate = df.groupby('CPTCode')['label_count'].mean()
df['cpt_denial_rate'] = df['CPTCode'].map(cpt_rate)

In [26]:
print(df.columns)

Index(['Clinic', 'TPCLIID', 'LIATPCLIid', 'ServiceDt', 'Service', 'ClaimID',
       'AmountCharged', 'CPTCode', 'ClientID', 'ClaimBillDate', 'Payer',
       'Provider', 'BillingProviderNPI', 'ClaimFacilityNPI', 'AuthStatus',
       'eligStatus', 'DenialFlag', 'lastActDt', 'cliANSI1', 'cliANSI2',
       'target1', 'target2', 'target3', 'target4', 'TotalPaid', 'TotalAdj',
       'TotalVoid', 'CoPay', 'Deduc', 'CoIns', 'CltResp', 'Balance',
       'MultiFlag', 'SameDayCli', 'DaysBetServiceToBilling',
       'tpcliStrModifier', 'tpcliStrPOS', 'f21diag1', 'f11insdob', 'labels',
       'label_count', 'cpt_denial_rate'],
      dtype='object')


In [27]:
df = df[df['label_count'] > 0]

clientid

In [28]:
df_sample['ClientID'].nunique()

1045

action for above one::Use Group Split by ClientID

Each client has MANY rows

Approx:

100,000 rows / 1,045 clients ≈ 95 rows per client

In [30]:
df_sample.groupby('Payer')['label_count'].mean().sort_values(ascending=False).head(10)

Payer
PAY_90206459    3.000000
PAY_40056146    2.500000
PAY_43328415    2.200000
PAY_55963010    2.000000
PAY_34850980    2.000000
PAY_41347409    2.000000
PAY_1638498     2.000000
PAY_6886777     2.000000
PAY_15827722    2.000000
PAY_37327686    1.857143
Name: label_count, dtype: float64

action required for payber above

In [36]:
df = df.copy()

df.loc[:, 'payer_denial_rate'] = df['Payer'].map(payer_rate)
df.loc[:, 'log_amount'] = np.log1p(df['AmountCharged'])

df.loc[:, 'tpcliStrModifier'] = df['tpcliStrModifier'].fillna('missing')
df.loc[:, 'f11insdob'] = df['f11insdob'].fillna('missing')

amount charged

In [32]:
df_sample.groupby('label_count')['AmountCharged'].mean()

label_count
0     85.092435
1     78.266490
2     82.963913
3    130.676957
4    192.560000
Name: AmountCharged, dtype: float64

action for amount charged


In [33]:
df['log_amount'] = np.log1p(df['AmountCharged'])

C:\Users\SreejaChiluveru\AppData\Local\Temp\ipykernel_3744\831339282.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['log_amount'] = np.log1p(df['AmountCharged'])


filling missing values

In [35]:
df['tpcliStrModifier'] = df['tpcliStrModifier'].fillna('missing')
df['f11insdob'] = df['f11insdob'].fillna('missing')

C:\Users\SreejaChiluveru\AppData\Local\Temp\ipykernel_3744\2056789386.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tpcliStrModifier'] = df['tpcliStrModifier'].fillna('missing')
C:\Users\SreejaChiluveru\AppData\Local\Temp\ipykernel_3744\2056789386.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['f11insdob'] = df['f11insdob'].fillna('missing')
